# Query Understanding

The first three demos closed the recall gap on the document side: one token per
concept, phrases identified, compounds split. The chapter opener's other failure
remains. The query "Bücher von Goethe" arrives as three tokens and the retriever
has no idea that "Bücher" and "books" are the same concept, that "Goethe" names a
person, or that the whole thing is a request for works by a specific author. This
demo adds the tools that pull that structure out of free text: word-relationship
expansion, part-of-speech tagging, and named entity recognition. It ends with the
practical concern that comes before all of them: a query that arrives misspelled
has to be repaired first.

**Learning goals:**

* Expand a query with WordNet synonyms and see the polysemy noise that comes with it
* Walk the WordNet hypernym and hyponym hierarchy for faceted and weighted expansion
* Read spaCy POS tags and use them to disambiguate homonyms and filter stop words
* Recognize named entities and map each entity type to a routing decision
* Chunk noun phrases with a rule-based grammar and read a spaCy dependency parse
* Implement edit distance and Soundex from scratch and repair a misspelled query
* Separate query expansion from query rewriting and rewrite the opener's F1 query

**Prerequisites:** Sections 3.1-3.3 (tokenization, normalization, phrases and compounds)

In [1]:
import re
import nltk
import spacy
from nltk.corpus import wordnet
from shared.display import print_table, display_md
from shared.text import tokenize

In [2]:
# One-time data/model downloads (quiet, safe to re-run)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

nlp = spacy.load("en_core_web_sm")

## 1. Word relationships: synonyms and homonyms

Every retrieval model so far has treated normalized tokens as independent
dimensions, with no notion that two of them might mean the same thing or that one
might mean two things. Real language breaks that in two directions. A **synonym**
is two words for one concept ("buy" and "purchase"); a **homonym** is one word for
two concepts ("bank" the institution and "bank" the riverside). Stemming and
lemmatization help with neither.

Synonyms break recall: a user searching for "purchase history" misses documents
indexed under "buy" or "buying". The classical fix attaches a synonym list to each
token, and WordNet is the standard source for English.

In [3]:
def wordnet_synonyms(word: str) -> list[str]:
    """All lemma names sharing a synset with `word`, minus the word itself."""
    names = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            name = lemma.name().replace("_", " ")
            if name.lower() != word.lower():
                names.add(name)
    return sorted(names)

print_table(
    [[w, ", ".join(wordnet_synonyms(w))] for w in ["buy", "fast", "big"]],
    headers=["Word", "WordNet synonyms (all senses)"],
)

| Word   | WordNet synonyms (all senses)                                                                                                                                                                                                                                                                                                                                                   |
|:-------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| buy    | bargain, bribe, corrupt, grease one's palms, purchase, steal                                                                                                                                                                                                                                                                                                                    |
| fast   | debauched, degenerate, degraded, dissipated, dissolute, fasting, firm, flying, immobile, libertine, loyal, profligate, quick, riotous, tight, truehearted                                                                                                                                                                                                                       |
| big    | adult, bad, bighearted, boastful, boastfully, bounteous, bountiful, braggart, bragging, braggy, cock-a-hoop, crowing, enceinte, expectant, freehanded, full-grown, fully grown, giving, gravid, great, grown, grownup, handsome, heavy, large, liberal, magnanimous, openhanded, prominent, self-aggrandising, self-aggrandizing, swelled, vainglorious, vauntingly, with child |

WordNet finds the useful synonym ("buy" gives "purchase") but also drags in senses
from unrelated meanings: "buy" also returns "bribe", "corrupt", and "steal", because
one of its synsets is the "bribe" sense. Expanding a query with all of these hurts
precision. Two ways to control it: restrict the synsets by part of speech (Section 3),
or weight the added terms below the original ones so an exact match always ranks first.

WordNet only knows linguistic synonyms. Domain shorthand is a different problem:
the opener's F1 query needs to match documents that call the same event a "Grand
Prix" and never use the tokens "F1" or "race". No lexical database captures that.
The fix is a hand-maintained synonym list owned by the search team.

In [4]:
DOMAIN_SYNONYMS = {
    "f1": ["Formula 1", "Formula One", "Grand Prix"],
    "race": ["Grand Prix", "GP"],
    "iphone": ["Apple iPhone", "iPhones"],
    "laptop": ["notebook"],
}

def expand_domain(query: str) -> list[str]:
    terms = tokenize(query)
    expanded = list(terms)
    for t in terms:
        expanded += [s.lower() for s in DOMAIN_SYNONYMS.get(t, [])]
    return sorted(set(expanded))

print_table(
    [[q, ", ".join(expand_domain(q))] for q in ["f1 race", "cheap laptop"]],
    headers=["Query", "After domain-synonym expansion"],
)

| Query        | After domain-synonym expansion                   |
|:-------------|:-------------------------------------------------|
| f1 race      | f1, formula 1, formula one, gp, grand prix, race |
| cheap laptop | cheap, laptop, notebook                          |

Homonyms are the harder direction. A bare query for "bank" cannot know whether the
user means the institution or the riverside. The words around a homonym usually
fix its sense, and the simplest signal is part of speech: "lead" the noun (the
metal) and "lead" the verb (to guide) carry different tags.

In [5]:
homonyms = [
    ("The old pipes still contain lead.", "lead"),
    ("She will lead the research team.", "lead"),
]
rows = []
for sentence, target in homonyms:
    tok = [t for t in nlp(sentence) if t.text.lower() == target][0]
    rows.append([sentence, tok.text, tok.pos_, tok.tag_])
print_table(rows, headers=["Sentence", "Word", "POS", "Fine tag"])

| Sentence                          | Word   | POS   | Fine tag   |
|:----------------------------------|:-------|:------|:-----------|
| The old pipes still contain lead. | lead   | NOUN  | NN         |
| She will lead the research team.  | lead   | VERB  | VB         |

**Caution: disambiguation needs context.** Both the classical and the modern fix
read the words around the homonym, so both fail when the query is too short to
supply any. A bare "bank" gives the system nothing, and the only honest responses
are to return both senses or to ask "did you mean...". Genuine ambiguity with no
context is inherent to language.

## 2. Hypernyms and hyponyms

WordNet also organizes nouns into a hierarchy. A **hypernym** is a broader category
("animal" is a hypernym of "cat") and a **hyponym** is a narrower one ("kitten" is
a hyponym of "cat"). Every English noun sits somewhere in that tree.

In [6]:
for syn in wordnet.synsets("cat", pos="n")[:1]:
    display_md(
        f"**`{syn.name()}`** -> {syn.definition()}\n\n"
        f"- hypernyms (broader): {[h.name() for h in syn.hypernyms()]}\n"
        f"- hyponyms (narrower): {[h.name() for h in syn.hyponyms()]}"
    )

**`cat.n.01`** -> feline mammal usually having thick soft fur and no ability to roar: domestic cats; wildcats

- hypernyms (broader): ['feline.n.01']
- hyponyms (narrower): ['domestic_cat.n.01', 'wildcat.n.03']

The hierarchy drives two retrieval moves. Going **down** the tree expands a query
with weighted hyponyms: a search for "cat" silently adds narrower kinds at a lower
weight, so documents about specific breeds surface for the general query. Going
**up** the tree powers faceted navigation: too few results for "cats" offers the
parent facet "mammals" that broadens the search.

In [7]:
def hyponym_expansion(word: str, depth: int = 2, max_syn: int = 1) -> list[str]:
    """Narrower terms below `word` in the WordNet noun hierarchy."""
    out = set()
    for syn in wordnet.synsets(word, pos="n")[:max_syn]:
        frontier = [syn]
        for _ in range(depth):
            nxt = []
            for s in frontier:
                for h in s.hyponyms():
                    out.add(h.lemmas()[0].name().replace("_", " "))
                    nxt.append(h)
            frontier = nxt
    return sorted(out)

print_table(
    [[w, len(hyponym_expansion(w)), ", ".join(hyponym_expansion(w)[:8]) + " ..."]
     for w in ["cat", "dog"]],
    headers=["Query term", "Narrower terms", "Sample (added at lower weight)"],
)

| Query term   |   Narrower terms | Sample (added at lower weight)                                                                                 |
|:-------------|-----------------:|:---------------------------------------------------------------------------------------------------------------|
| cat          |               29 | Abyssinian, Angora, Burmese cat, Egyptian cat, European wildcat, Maltese, Manx, Persian cat ...                |
| dog          |               60 | Brabancon griffon, Cardigan, Chihuahua, Eskimo dog, Great Dane, Great Pyrenees, Japanese spaniel, Leonberg ... |

## 3. Part-of-speech tagging

A **part-of-speech tagger** labels each token with its grammatical class, using the
surrounding words to disambiguate a surface form that could belong to several
classes. Modern taggers run a small neural network over the token sequence; this is
the default in spaCy. Take the opener's person-lookup query.

In [8]:
doc = nlp("Who is Albert Einstein?")
print_table(
    [[t.text, t.pos_, t.tag_, t.dep_] for t in doc],
    headers=["Token", "POS (universal)", "Tag (Penn)", "Dependency"],
)

| Token    | POS (universal)   | Tag (Penn)   | Dependency   |
|:---------|:------------------|:-------------|:-------------|
| Who      | PRON              | WP           | attr         |
| is       | AUX               | VBZ          | ROOT         |
| Albert   | PROPN             | NNP          | compound     |
| Einstein | PROPN             | NNP          | nsubj        |
| ?        | PUNCT             | .            | punct        |

Two tag sets appear. The `POS` column is the coarse Universal set (around 17 categories: NOUN, VERB, PROPN, PRON, AUX, ...); the `Tag` column is the fine-grained Penn Treebank set (around 45 categories: `VBZ` third-person verb, `NNP` proper noun, `WP` WH-pronoun). The pattern here (a WH-pronoun, a copula verb, and a two-token proper noun) is exactly what marks this as a person-lookup question rather than a keyword search. Section 5 (intent routing) acts on it.

POS tagging earns its place in the pipeline through three retrieval-side uses. The
first is disambiguating a surface form that is a noun in one query and a verb in
another.

In [9]:
run_cases = [
    ("That was a good run.", "run"),
    ("I run five miles daily.", "run"),
]
rows = [[s, w, *[(t.pos_, t.tag_) for t in nlp(s) if t.text.lower() == w][0]]
        for s, w in run_cases]
print_table(rows, headers=["Sentence", "Word", "POS", "Fine tag"])

| Sentence                | Word   | POS   | Fine tag   |
|:------------------------|:-------|:------|:-----------|
| That was a good run.    | run    | NOUN  | NN         |
| I run five miles daily. | run    | VERB  | VBP        |

The second is POS-aware stop-word filtering: not every "it" is a pronoun. In "the
IT department" it is a proper noun and must be kept; in "it is easy" it is a
function word and can be dropped. Tagging tells the two apart, which is how the "IT
security" case from the stop-word discussion is handled correctly.

In [10]:
it_cases = ["the IT department", "it is easy"]
rows = []
for phrase in it_cases:
    tok = [t for t in nlp(phrase) if t.text.lower() == "it"][0]
    keep = tok.pos_ in ("PROPN", "NOUN")
    rows.append([phrase, tok.text, tok.pos_, "keep" if keep else "drop as stop word"])
print_table(rows, headers=["Phrase", "Token", "POS", "Filtering decision"])

| Phrase            | Token   | POS   | Filtering decision   |
|:------------------|:--------|:------|:---------------------|
| the IT department | IT      | PROPN | keep                 |
| it is easy        | it      | PRON  | drop as stop word    |

The third is disambiguating the lemmatizer: WordNet needs the POS tag to return the
right base form.

In [11]:
from nltk.stem import WordNetLemmatizer
wn = WordNetLemmatizer()
print_table(
    [["meeting", "n (noun)", wn.lemmatize("meeting", "n")],
     ["meeting", "v (verb)", wn.lemmatize("meeting", "v")]],
    headers=["Word", "POS given", "WordNet lemma"],
)

| Word    | POS given   | WordNet lemma   |
|:--------|:------------|:----------------|
| meeting | n (noun)    | meeting         |
| meeting | v (verb)    | meet            |

## 4. Named entity recognition

The two proper nouns "Albert Einstein" are not just proper nouns; they name a
specific person. **Named entity recognition (NER)** classifies spans of tokens into
types (person, location, organization, money, date) and runs right after POS
tagging. spaCy's model tags a full sentence in one pass.

In [12]:
doc = nlp("Jack Higgins deposits £50,000 with BestBank in London.")
print_table(
    [[ent.text, ent.label_, spacy.explain(ent.label_)] for ent in doc.ents],
    headers=["Entity", "Label", "Meaning"],
)

| Entity       | Label   | Meaning                                 |
|:-------------|:--------|:----------------------------------------|
| Jack Higgins | PERSON  | People, including fictional             |
| 50,000       | MONEY   | Monetary values, including unit         |
| BestBank     | ORG     | Companies, agencies, institutions, etc. |
| London       | GPE     | Countries, cities, states               |

Every entity type exposes a routing decision, and four types cover most of the
retrieval-side use cases. The router itself is the next section's subject; here we
only extract the entities and note where each one would send the query.

In [13]:
routing_queries = [
    ("Who is Albert Einstein?", "PERSON", "person lookup: Wikipedia, biography DBs"),
    ("Who won the F1 race last weekend?", "DATE", "restrict to recent news, rank fresh"),
    ("What to do in Basel?", "GPE", "regional content, attach a map"),
    ("Where can I buy the latest iPhone?", "PRODUCT/ORG", "boost shopping, price widget"),
]
rows = []
for query, expected, route in routing_queries:
    ents = ", ".join(f"{e.text} [{e.label_}]" for e in nlp(query).ents) or "(none)"
    rows.append([query, ents, route])
print_table(rows, headers=["Query", "Entities found", "Routing decision"])

| Query                              | Entities found                | Routing decision                        |
|:-----------------------------------|:------------------------------|:----------------------------------------|
| Who is Albert Einstein?            | Albert Einstein [PERSON]      | person lookup: Wikipedia, biography DBs |
| Who won the F1 race last weekend?  | F1 [GPE], last weekend [DATE] | restrict to recent news, rank fresh     |
| What to do in Basel?               | Basel [GPE]                   | regional content, attach a map          |
| Where can I buy the latest iPhone? | (none)                        | boost shopping, price widget            |

The model does not always find the category we expect (short queries starve it of context, and `iPhone` is often missed as a product), which is exactly why the next section adds a trained classifier on top rather than trusting NER alone.

NER also generates candidate phrases for free: consecutive tokens tagged `PERSON` form a multi-token unit like "Albert Einstein" that should be indexed whole, the same phrase-detection goal as the previous demo but more targeted.

## 5. Chunking and dependencies

For queries that do not fit a POS-plus-NER analysis, a lightweight **chunker**
groups consecutive tokens matching a small grammar. The classic noun-phrase pattern
is `NP -> DET? ADJ* NOUN`, which `nltk.RegexpParser` implements cheaply. The `<NN.*>`
pattern below also catches plural and proper nouns.

In [14]:
grammar = r"NP: {<DT>?<JJ>*<NN.*>+}"
chunker = nltk.RegexpParser(grammar)

def noun_phrases(text: str) -> list[str]:
    tree = chunker.parse(nltk.pos_tag(nltk.word_tokenize(text)))
    return [" ".join(w for w, _ in st.leaves())
            for st in tree.subtrees() if st.label() == "NP"]

print_table(
    [[s, ", ".join(noun_phrases(s))]
     for s in ["The quick brown fox jumps over the lazy dog.",
               "a red car"]],
    headers=["Sentence", "Noun phrases"],
)

| Sentence                                     | Noun phrases                      |
|:---------------------------------------------|:----------------------------------|
| The quick brown fox jumps over the lazy dog. | The quick brown fox, the lazy dog |
| a red car                                    | a red car                         |

A full dependency parser goes further: it connects each word to its grammatical
head, so we can read off what modifies what. spaCy produces this once POS tags are
available, and it is more reliable than the rule-based chunker on messy queries.

In [15]:
query = "cheap hotels near the Eiffel Tower with free wifi"
doc = nlp(query)
print_table(
    [[t.text, t.pos_, t.dep_, t.head.text] for t in doc],
    headers=["Token", "POS", "Dependency", "Head"],
)

| Token   | POS   | Dependency   | Head   |
|:--------|:------|:-------------|:-------|
| cheap   | ADJ   | amod         | hotels |
| hotels  | NOUN  | ROOT         | hotels |
| near    | ADP   | prep         | hotels |
| the     | DET   | det          | Tower  |
| Eiffel  | PROPN | compound     | Tower  |
| Tower   | PROPN | pobj         | near   |
| with    | ADP   | prep         | hotels |
| free    | ADJ   | amod         | wifi   |
| wifi    | NOUN  | pobj         | with   |

In [16]:
root = [t for t in doc if t.dep_ == "ROOT"][0]
modifiers = [t.text for t in doc if t.head == root and t.dep_ in ("amod", "compound")]
prep_phrases = [(t.text, " ".join(c.text for c in t.subtree)) for t in doc if t.dep_ == "prep"]

display_md(
    f"**Structured reading of the query:**\n\n"
    f"- Head noun: **{root.text}** (this is what the user is searching for)\n"
    f"- Modifiers on the head: {modifiers}\n"
    f"- Prepositional constraints: {prep_phrases}\n\n"
    f"The parse turns a flat string into a search for *{root.text}*, filtered by "
    f"*cheap*, constrained near the *Eiffel Tower*, requiring *free wifi*."
)

**Structured reading of the query:**

- Head noun: **hotels** (this is what the user is searching for)
- Modifiers on the head: ['cheap']
- Prepositional constraints: [('near', 'near the Eiffel Tower'), ('with', 'with free wifi')]

The parse turns a flat string into a search for *hotels*, filtered by *cheap*, constrained near the *Eiffel Tower*, requiring *free wifi*.

## 6. Spell correction and "did you mean?"

Every query pipeline has to handle typos. A user typing "Goehte" (two letters
transposed) gets nothing if the retriever demands an exact match on "Goethe". The
classical tool is **edit distance**: the minimum number of edits to turn one string
into another. Plain Levenshtein counts insertions, deletions, and substitutions;
the Damerau variant adds transposition of adjacent letters, which is the single
most common typing error.

In [17]:
def levenshtein(a: str, b: str) -> int:
    """Insertions, deletions, substitutions."""
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[m][n]

def damerau_levenshtein(a: str, b: str) -> int:
    """Levenshtein plus transposition of two adjacent characters."""
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                dp[i][j] = min(dp[i][j], dp[i - 2][j - 2] + 1)
    return dp[m][n]

In [18]:
pairs = [("goehte", "goethe"), ("recieve", "receive"), ("teh", "the"),
         ("britny", "britney"), ("kitten", "sitting")]
print_table(
    [[a, b, levenshtein(a, b), damerau_levenshtein(a, b)] for a, b in pairs],
    headers=["Typed", "Target", "Levenshtein", "Damerau-Levenshtein"],
)

| Typed   | Target   |   Levenshtein |   Damerau-Levenshtein |
|:--------|:---------|--------------:|----------------------:|
| goehte  | goethe   |             2 |                     1 |
| recieve | receive  |             2 |                     1 |
| teh     | the      |             2 |                     1 |
| britny  | britney  |             1 |                     1 |
| kitten  | sitting  |             3 |                     3 |

<div style="border-left: 4px solid #0284C7; background: rgba(2, 132, 199, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#0284C7; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Info</strong><br>
The transposition typos <code>goehte</code>, <code>recieve</code>, <code>teh</code> all cost <strong>2</strong> under plain Levenshtein but only <strong>1</strong> under Damerau, because a swap of adjacent letters is one edit for a human and two for the plain metric. Ranking corrections by Damerau distance puts the intended word first for the most common kind of typo.
</div>

With the metric in hand, correction is a search over a dictionary for the
lowest-distance real word. This is the "did you mean?" box.

In [19]:
DICTIONARY = [
    "goethe", "schiller", "kafka", "information", "retrieval", "receive",
    "believe", "search", "query", "document", "the", "book", "books",
]

def did_you_mean(word: str, max_dist: int = 2) -> list[tuple[str, int]]:
    cands = [(w, damerau_levenshtein(word.lower(), w)) for w in DICTIONARY]
    cands = [(w, d) for w, d in cands if 0 < d <= max_dist]
    return sorted(cands, key=lambda x: x[1])[:3]

rows = [[w, ", ".join(f"{c} (d={d})" for c, d in did_you_mean(w)) or "(no suggestion)"]
        for w in ["goehte", "retreival", "recieve", "bok"]]
print_table(rows, headers=["Misspelled", "Top suggestions"])

| Misspelled   | Top suggestions              |
|:-------------|:-----------------------------|
| goehte       | goethe (d=1)                 |
| retreival    | retrieval (d=1)              |
| recieve      | receive (d=1), believe (d=2) |
| bok          | book (d=1), books (d=2)      |

Edit distance cannot fix a name it has never seen and cannot group spellings that
sound alike but differ in many letters. **Phonetic codes** solve the second problem
by reducing a word to a short code of its rough pronunciation. Soundex is the
classic: same-sounding names collapse to one code and become candidates for each
other.

In [20]:
def soundex(name: str) -> str:
    """4-character Soundex code: first letter plus three consonant digits."""
    name = re.sub(r"[^A-Za-z]", "", name).upper()
    if not name:
        return ""
    digit = {**dict.fromkeys("BFPV", "1"), **dict.fromkeys("CGJKQSXZ", "2"),
             **dict.fromkeys("DT", "3"), "L": "4",
             **dict.fromkeys("MN", "5"), "R": "6"}
    code = name[0]
    prev = digit.get(name[0], "")
    for ch in name[1:]:
        d = digit.get(ch, "")
        if d and d != prev:
            code += d
        if ch not in "HW":
            prev = d
    return (code + "000")[:4]

names = ["Smith", "Smyth", "Britney", "Britny", "Brittney", "Britnee"]
print_table([[n, soundex(n)] for n in names], headers=["Name", "Soundex code"])

| Name     | Soundex code   |
|:---------|:---------------|
| Smith    | S530           |
| Smyth    | S530           |
| Britney  | B635           |
| Britny   | B635           |
| Brittney | B635           |
| Britnee  | B635           |

`Smith` and `Smyth` share `S530`; every spelling of `Britney` shares `B635`. That is what lets a search for one variant surface documents that use another. The full strategy runs on both sides: at indexing time keep the original spelling **and** add the corrected form, so a document reads whether the user types the right or the wrong spelling.

**Caution: names resist canonicalization.** Rewriting every `Britny` and `Brittney` to one `Britney` merges distinct real people, because several of those variants are also legitimate names in their own right. The safe move is to keep the original in the index and, at query time, expand to the variants rather than silently overwrite.

## 7. Query expansion vs query rewriting

Nearly every transformation in this demo takes the user's tokens and changes them so
the query matches more of the collection. The IR literature splits them into two
families.

| Technique | Family | Effect on the query |
| --- | --- | --- |
| Synonym expansion | Expansion | keeps 'buy', adds 'purchase' at lower weight |
| Weighted hyponyms | Expansion | keeps 'cat', adds 'siamese', 'persian' |
| Name-variant add | Expansion | keeps 'Britney', adds 'Britny', 'Brittney' |
| Spell correction | Rewriting | 'Goehte' -> 'Goethe' |
| Alias substitution | Rewriting | 'F1' -> 'Formula 1' |
| Temporal normalization | Rewriting | 'last weekend' -> a date range |
| Compound splitting | Rewriting | 'Wolkenkratzer' -> 'Wolken', 'Kratzer' |

**Key insight:** the line blurs. A rewrite that adds tokens alongside the originals (`F1 OR "Formula 1" OR "Grand Prix"`) is expansion in disguise; an expansion that later drops the originals is a rewrite. What matters is that the retriever sees a query different from the one the user typed.

Everything in this demo comes together on the opener's F1 query. Rewriting it means
dropping the non-content question words (using POS tags), expanding the domain
shorthand (using the hand-maintained list), and normalizing the time expression.

In [21]:
def rewrite_query(query: str) -> dict:
    doc = nlp(query)
    # 1. Drop WH-pronouns, auxiliaries, and the main verb: they carry no content.
    content = [t for t in doc if t.pos_ not in ("PRON", "AUX", "VERB", "PUNCT", "DET", "ADP")]
    kept = [t.text.lower() for t in content]
    # 2. Expand domain aliases on the kept content tokens.
    expanded = []
    for t in kept:
        expanded.append(t)
        expanded += [s.lower() for s in DOMAIN_SYNONYMS.get(t, [])]
    # 3. Normalize the temporal expression from any DATE entity.
    has_date = any(e.label_ == "DATE" for e in doc.ents) or "weekend" in query.lower()
    return {
        "content_tokens": kept,
        "expanded": sorted(set(expanded)),
        "date_filter": "published: last 7 days" if has_date else "(none)",
    }

result = rewrite_query("Who won the F1 race on the weekend?")
display_md(
    "**Rewriting** `Who won the F1 race on the weekend?`\n\n"
    f"- Content tokens after dropping question words: {result['content_tokens']}\n"
    f"- After domain-alias expansion: {result['expanded']}\n"
    f"- Temporal constraint: `{result['date_filter']}`\n\n"
    "What reaches the retriever is a scoped keyword query, not a question. Classical "
    "rewriting stops here: it can retrieve the right race report but cannot produce the "
    "one-word answer \"Verstappen\". That last step needs a reader or generator (the "
    "retrieval-augmented generation chapter)."
)

**Rewriting** `Who won the F1 race on the weekend?`

- Content tokens after dropping question words: ['f1', 'race', 'weekend']
- After domain-alias expansion: ['f1', 'formula 1', 'formula one', 'gp', 'grand prix', 'race', 'weekend']
- Temporal constraint: `published: last 7 days`

What reaches the retriever is a scoped keyword query, not a question. Classical rewriting stops here: it can retrieve the right race report but cannot produce the one-word answer "Verstappen". That last step needs a reader or generator (the retrieval-augmented generation chapter).

## Summary

| Technique | Tool | Retrieval benefit | Cost / risk |
| --- | --- | --- | --- |
| Synonym expansion | WordNet or domain list | Recall on wording variants | Polysemy noise |
| Hyponym expansion | WordNet hierarchy, weighted | Recall on narrower terms | Topic drift |
| POS tagging | spaCy neural tagger | Homonyms, stop-word filtering, lemma disambiguation | Needs surrounding context |
| NER | spaCy span classifier | Routing, entity phrases | Misses out-of-context entities |
| Chunking / dependency | RegexpParser, spaCy parser | Head noun, modifiers, constraints | Grammar is brittle |
| Edit distance | Damerau-Levenshtein | Typo repair, did-you-mean | Slow over a full lexicon |
| Phonetic code | Soundex | Same-sounding name variants | English-centric, coarse |

<div style="border-left: 4px solid #C8102E; background: rgba(200, 16, 46, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#C8102E; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Takeaway</strong><br>
Query understanding turns a bag of characters into structured intent. Each layer (spelling repair, POS, NER, expansion) unlocks a different capability, and each produces a discrete output that a downstream system can act on. That is why POS tagging, NER, and spell correction survive even in fully neural stacks: they are cheap and their outputs are structured. The next demo (ch03-05) takes exactly these outputs (language, POS, entities, corrections) and turns them into a single routing decision.
</div>

## Try it yourself

1. Expand the query "python programming" with `wordnet_synonyms`. Does the snake
   sense of "python" pollute the result, and would a POS filter remove it?
2. Add a keyboard-aware variant of `damerau_levenshtein` where substituting adjacent
   QWERTY keys (for example `q`/`w`) costs less than a distant one.
3. Add a new alias to `DOMAIN_SYNONYMS` and rewrite a query that uses it.

In [22]:
query = "python programming"
lines = []
for word in tokenize(query):
    syns = wordnet_synonyms(word)
    lines.append(f"- **{word}** -> {syns[:10] if syns else '(no synonyms)'}")
display_md(f"**Synonym expansion of `{query}`:**\n\n" + "\n".join(lines))

**Synonym expansion of `python programming`:**

- **python** -> (no synonyms)
- **programming** -> ['computer programing', 'computer programming', 'program', 'programing', 'programme', 'scheduling']